In [ ]:
import os
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""

In [ ]:
%pip install --no-build-isolation --force-reinstall \
    "boto3>=1.28.57" \
    "awscli>=1.29.57" \
    "botocore>=1.31.57"

     |████████████████████████████████| 135 kB 17.7 MB/s 
     |████████████████████████████████| 4.3 MB 33.8 MB/s 
     |████████████████████████████████| 11.3 MB 94.2 MB/s 
     |████████████████████████████████| 79 kB 79.4 MB/s 
     |████████████████████████████████| 738 kB 77.7 MB/s 
     |████████████████████████████████| 548 kB 81.0 MB/s 
     |████████████████████████████████| 143 kB 107.4 MB/s 
     |████████████████████████████████| 247 kB 101.4 MB/s 
     |████████████████████████████████| 83 kB 77.8 MB/s 


  Attempting uninstall: jmespath
    Found existing installation: jmespath 1.0.1
    Uninstalling jmespath-1.0.1:
      Successfully uninstalled jmespath-1.0.1
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.26.18
    Uninstalling urllib3-1.26.18:
      Successfully uninstalled urllib3-1.26.18
  Attempting uninstall: six
    Found existing installation: six 1.16.0
    Uninstalling six-1.16.0:
      Successfully uninstalled six-1.16.0
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.2
    Uninstalling python-dateutil-2.8.2:
      Successfully uninstalled python-dateutil-2.8.2
  Attempting uninstall: botocore
    Found existing installation: botocore 1.31.73
    Uninstalling botocore-1.31.73:
      Successfully uninstalled botocore-1.31.73
  Attempting uninstall: s3transfer
    Found existing installation: s3transfer 0.7.0
    Uninstalling s3transfer-0.7.0:
      Successfully uninstalled s3transfer-0.7.0
  Attempting 

    Uninstalling awscli-1.29.73:
      Successfully uninstalled awscli-1.29.73
ERROR: After October 2020 you may experience errors when installing or updating packages. This is because pip will change the way that it resolves dependency conflicts.
We recommend you use --use-feature=2020-resolver to test your packages with the new resolver before it becomes the default.
datarobot-drum 1.10.6 requires datarobot==3.1.0, but you'll have datarobot 3.2.1 which is incompatible.
datarobot-drum 1.10.6 requires Pillow<=9.3.0, but you'll have pillow 10.0.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install datarobotx[llm] datarobot-mlops datarobot-mlops-connected-client transformers

In [ ]:
!pip install py-readability-metrics nltk

In [ ]:
!pip3 install python-docx PyPDF2

In [ ]:
import json
import os

import boto3

import docx

In [ ]:
def load_model(company_profile_path, job_description_path, candidate_profile_path):

    import docx
    import PyPDF2 
    
    def read_file(file_path):
        if file_path is None:
            return "No file path provided."
    
        # Check file extension to determine processing method
        file_extension = file_path.split(".")[-1].lower()
    
        if file_extension == "txt":
            # Read text files
            with open(file_path, "r", encoding="utf-8") as file:
                text = file.read()
        elif file_extension == "docx":
            # Process Word documents
            doc = docx.Document(file_path)
            text = " ".join([p.text for p in doc.paragraphs])
        elif file_extension == "pdf":
            # Process PDFs
            with open(file_path, "rb") as file:
                pdf_reader = PyPDF2.PdfFileReader(file)
                text = ""
                for page_num in range(pdf_reader.numPages):
                    page = pdf_reader.getPage(page_num)
                    text += page.extractText()
        else:
            return f"File type {file_extension} not supported."
    
        return text

    comapny_profile = read_file(company_profile_path)
    job_description = read_file(job_description_path)
    candidate_profile = read_file(candidate_profile_path)

    return comapny_profile, job_description, candidate_profile

In [ ]:
def score_unstructured(comapny_profile, job_description, candidate_profile, goals, parameters):
    import boto3
    import json

    #Create the connection to Bedrock
    bedrock_runtime = boto3.client(
        service_name='bedrock-runtime',
        region_name='us-east-1'
    )

    modelId = "anthropic.claude-v2"
    accept = "*/*"
    contentType = "application/json"

    prompt_data = """You are a seasoned digital marketing manager at NextGen Digital Solutions, bringing over a decade of experience in SEO, PPC, 
    and content strategy to the table. At 35 years old, you have spent the last three years nurturing and expanding the company’s digital marketing department, 
    demonstrating a keen eye for talent and potential. With a Bachelor’s Degree in Business Administration, specializing in Marketing, from the University of 
    Texas at Austin, you combine their educational background with practical experience to lead their team effectively. You are known for being meticulous, analytical, 
    and having a strong ability to identify candidates’ potential through detailed and insightful interviews. Despite your high expectations and the value they place on 
    preparation, you are approachable and value clear communication, creative problem-solving, and unwavering dedication in potential candidates. Outside the professional 
    realm, you immerse yourself in the latest technology trends and digital innovations, regularly attending industry conferences to ensure they are at the forefront of 
    digital marketing knowledge.
    """

    user_input = f"""The task is to create a tailored list of five interview questions that are intricately aligned with the 
    Company Profile: {comapny_profile}
    Job Description: {job_description}
    Candidate’s Profile: {candidate_profile}
    and the Goals of the Interview:{str(goals)} 
    This will ensure a holistic evaluation of the candidate, facilitating an in-depth understanding of their suitability for the SEO Specialist position at NextGen Digital Solutions.
    Also give acceptable answers in bullet points for refernence
    
    Please generate the response in a json format, the keys of the jason should be
    question number, question, answer, alinged goal
    """

    body = json.dumps({**{"prompt": f"\n\nHuman: {user_input}" + f"\n\n{prompt_data} \nAssistant:"}, **parameters})
    response = bedrock_runtime.invoke_model(
        body=body, modelId=modelId, accept=accept, contentType=contentType
    )
    answer = json.loads(response.get("body").read())
    return answer

In [ ]:
def get_completion(company_profile_path, job_description_path, candidates_profile_path, goals):

    company_profile, job_description, candidate_profile = load_model(company_profile_path, job_description_path, candidates_profile_path)
    
    output = score_unstructured(
        company_profile,
        job_description,
        candidate_profile,
        goals = goals,
        parameters = parameters
    )

    return output

In [ ]:
parameters = {
    # "stopSequences":[],
    "max_tokens_to_sample": 1000,
    "temperature": 0,
    "top_p": 0.999,
    "top_k": 250,
    }

In [ ]:
modelId = "anthropic.claude-v2"
accept = "*/*"
contentType = "application/json"

In [ ]:
company_profile_path = '/home/notebooks/storage/NextGen Digital Solutions.docx'
job_description_path = '/home/notebooks/storage/SEO Specialist.docx'
candidates_profile_path = '/home/notebooks/storage/Jordan Taylor.docx'
goals = ["Technical Skill", "Learning Ability"]

In [ ]:
get_completion(company_profile_path, job_description_path, candidates_profile_path, goals)

{'completion': ' Here are 5 tailored interview questions for the SEO Specialist role at NextGen Digital Solutions, aligned with assessing the candidate\'s technical skills and learning ability:\n\n1. {\n  "questionNumber": 1,\n  "question": "Walk me through your process for conducting keyword research and competitive analysis. What tools and techniques do you use to identify high-potential keywords and phrases?",\n  "answer": "- Utilize keyword research tools like SEMrush, Moz, and Google Keyword Planner to generate keyword ideas and analyze search volume/competition. \n- Analyze competitors\' website content and metadata to identify keywords they are targeting.\n- Leverage Google Trends and search predictions to identify rising trends.  \n- Conduct long-tail keyword research to uncover niche, high-conversion keywords.\n- Analyze keyword difficulty using metrics like domain authority and backlinks. \n- Identify keyword gaps competitors are missing out on.\n- Organize keywords into prio

In [ ]:
import datarobotx as drx

!mkdir storage/deploy/

deployment = drx.deploy(
    "storage/deploy/",
    name="claude-v2_questions",
    hooks={
        "load_model": load_model, 
        "score_unstructured": score_unstructured
    },
    extra_requirements = ['python-docx', 'PyPDF2', 'boto3>=1.28.57'],
    environment_id="64d2ba178dd3f0b1fa2162f0",
)
# Enable storing prediction data, necessary for Data Export for monitoring purposes
deployment.dr_deployment.update_predictions_data_collection_settings(enabled=True)

# Deploying custom model
  - Unable to auto-detect model type; any provided paths and files will be
    exported - dependencies should be explicitly specified using
    `extra_requirements` or `environment_id`
  - Preparing model and environment...
  - Using environment [[DataRobot] Python 3.11 GenAI
    v1](https://app.datarobot.com/model-registry/custom-environments/64d2ba178dd3f0b1fa2162f0)
    for deployment
  - Configuring and uploading custom model...
    100%|██████████████████████████████████| 12.8k/12.8k [00:00<00:00, 6.26MB/s]
  - Registered custom model
    [claude-v2_questions](https://app.datarobot.com/model-registry/custom-models/653e97697dd711b0dd42e794/info)
    with target type: Unstructured
  - Installing additional dependencies...


  - Creating and deploying model package...


Exception raised when running coroutine deploy
Traceback (most recent call last):
  File "/etc/system/kernel/.venv/lib64/python3.9/site-packages/datarobotx/common/utils.py", line 115, in thread_entry_point
    return asyncio.run(async_entry_point())
  File "/usr/lib64/python3.9/asyncio/runners.py", line 44, in run
    return loop.run_until_complete(main)
  File "/usr/lib64/python3.9/asyncio/base_events.py", line 647, in run_until_complete
    return future.result()
  File "/etc/system/kernel/.venv/lib64/python3.9/site-packages/datarobotx/common/utils.py", line 109, in async_entry_point
    await task
  File "/etc/system/kernel/.venv/lib64/python3.9/site-packages/datarobotx/models/deploy.py", line 266, in deploy
    self.destination._deployment_id = await self.transmit()
  File "/etc/system/kernel/.venv/lib64/python3.9/site-packages/datarobotx/models/deploy.py", line 503, in transmit
    deploy_id = await deploy_client.deploy_from_package(package_id, model_id)  # type: ignore[arg-type]


ValueError: DataRobot error detected while polling the status of job id '943b909e-22a3-4e77-889a-c7197cf268a5'
    at URL 'https://app.datarobot.com/api/v2/status/943b909e-22a3-4e77-889a-c7197cf268a5/'. Status:
    ERROR Message: ERROR: Unable to load hook: load_model. Make sure your Python version matches the
    environment selected and that the needed requirements are included in the deploy call